# DỰ ĐOÁN GIÁ NHÀ VN — EDA & PREPROCESSING

**Assignment 03 — Neural Networks and Representation Learning**

**Bài toán:** Hồi quy (Regression) — dự đoán giá nhà

**Môi trường:** conda env `assignment2`

---

## 1. Import libraries & Load data

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import joblib

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_PATH = os.path.join('..', 'data', 'VN_housing_dataset.csv')
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=['Unnamed: 0'])
print('Raw shape:', df.shape)
df.head(3)

## 2. Tiền xử lý dữ liệu

Làm sạch text và chuyển các cột về dạng số.

In [ ]:
def extract_number(text):
    if pd.isna(text): return np.nan
    # extract numbers, replace comma with dot for decimals
    match = re.search(r'([\d.,]+)', str(text))
    if match:
        val = match.group(1).replace('.', '').replace(',', '.')
        try:
            return float(val)
        except:
            return np.nan
    return np.nan

# Xử lý cột Diện tích (Area), Giá/m2 (Price/m2)
df['Area'] = df['Diện tích'].apply(extract_number)
df['Price_per_m2'] = df['Giá/m2'].apply(extract_number)

# Tính giá tổng (Total Price) = Area * Price_per_m2 (triệu VNĐ)
df['TotalPrice'] = df['Area'] * df['Price_per_m2']

# Xử lý số tầng, số phòng ngủ, chiều dài, rộng
df['Floors'] = df['Số tầng'].apply(extract_number)
df['Bedrooms'] = df['Số phòng ngủ'].apply(extract_number)
df['Length'] = df['Dài'].apply(extract_number)
df['Width'] = df['Rộng'].apply(extract_number)

# Loại bỏ các cột text không cần thiết & đổi tên
cols_drop = ['Ngày', 'Địa chỉ', 'Diện tích', 'Dài', 'Rộng', 'Giá/m2', 'Số tầng', 'Số phòng ngủ']
df = df.drop(columns=cols_drop)

df = df.rename(columns={
    'Quận': 'District',
    'Huyện': 'Ward',
    'Loại hình nhà ở': 'House_Type',
    'Giấy tờ pháp lý': 'Legal_Status'
})
print('Sau khi parse số:', df.shape)
df.head()

### 2.1 Xử lý Missing Values & Outliers

In [ ]:
# Xóa các hàng mất Target (TotalPrice)
df = df.dropna(subset=['TotalPrice', 'Area']).copy()

# Lọc Outliers cơ bản hợp lý hóa dữ liệu
df = df[(df['Area'] > 10) & (df['Area'] < 500)] # Diện tích 10m2 -> 500m2
df = df[(df['TotalPrice'] > 100) & (df['TotalPrice'] < 50000)] # Giá 100tr -> 50 tỷ

# Lấp đầy NaN với median đối với các feature số
for c in ['Floors', 'Bedrooms', 'Length', 'Width']:
    df[c] = df[c].fillna(df[c].median())

# Lấp đầy Categorical
df['Legal_Status'] = df['Legal_Status'].fillna('Unknown')

print('Sau khi clean up:', df.shape)
df.isnull().sum()

## 3. Khám phá (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.histplot(df['TotalPrice'], kde=True, ax=axes[0])
axes[0].set_title('Phân bố Giá Nhà (Triệu VNĐ)')

sns.histplot(np.log1p(df['TotalPrice']), kde=True, ax=axes[1])
axes[1].set_title('Phân bố Giá Nhà (Log Scale)')
plt.show()

# Vì phân bố lệch phải rất mạnh, ta sử dụng Log Transform trên Target
df['LogPrice'] = np.log1p(df['TotalPrice'])

## 4. Chuẩn bị Dữ liệu cho Mô hình (Encoding & Splitting)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Lọc lấy Top 15 Quận phổ biến (gom nhóm phần còn lại)
top_districts = df['District'].value_counts().nlargest(15).index
df['District'] = df['District'].apply(lambda x: x if x in top_districts else 'Other')

# Get Dummies
df_encoded = pd.get_dummies(df.drop(columns=['Price_per_m2', 'TotalPrice', 'Ward']), columns=['District', 'House_Type', 'Legal_Status'], drop_first=True)

X = df_encoded.drop(columns=['LogPrice'])
y = df_encoded['LogPrice']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Train:', X_train_scaled.shape)
print('Val:', X_val_scaled.shape)
print('Test:', X_test_scaled.shape)

## 5. Lưu kết cụ

In [ ]:
MODEL_DIR = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.pkl'))
joblib.dump(list(X.columns), os.path.join(MODEL_DIR, 'feature_names.pkl'))

np.savez_compressed(os.path.join(MODEL_DIR, 'preprocessed_data.npz'),
                    X_train=X_train_scaled, y_train=y_train.values,
                    X_val=X_val_scaled, y_val=y_val.values,
                    X_test=X_test_scaled, y_test=y_test.values)
print('✅ Đã lưu tiền xử lý.')